# Kilter Grade Prediction — Feature Engineering

The data-processing step combines **multi-hot encoding** of the `holds` and `role` features with the production of geometric features to enrich the dataset.

## 1. The cleaned dataset

In the cleaned dataset (see `01_exploratory_analysis.ipynb`), `holds` is a stringified list of `[placement_id, x, y, role_id]`, where role IDs are `12=start`, `13=hand`, `14=finish`, and `15=foot`. The dataset also includes other features relevant to predicting the target variable `difficulty_average`.

In [1]:
from pathlib import Path
import ast
import itertools
import sys
import warnings

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# When run from notebooks/, the project root is its parent.
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
CLEAN_CSV = PROJECT_ROOT / "data" / "processed" / "data_cleaned.csv"


In [2]:
# The cleaned file is produced by utils/data_preparation.py.
df = pd.read_csv(CLEAN_CSV)
print(f"Loaded {len(df):,} climbs and {df.shape[1]} columns")
print(df.columns.tolist())
display(df.head(2))

Loaded 11,516 climbs and 10 columns
['climb_uuid', 'name', 'angle', 'layout_id', 'n_holds', 'holds', 'difficulty_average', 'quality_average', 'ascensionist_count', 'is_nomatch']


,climb_uuid,name,angle,layout_id,n_holds,holds,difficulty_average,quality_average,ascensionist_count,is_nomatch
0,167e313e708e42e39796b521ca6eeec7,pogo a gogo,45,1,11,"[(1097, 64, 16, 15), (1137, 112, 32, 15), (114...",21.0,2.6,5,0
1,F1DFFDBB7EB94D65AADAA43B7D817836,foot secret,0,1,14,"[(1076, 112, 8, 15), (1079, 88, 8, 15), (1137,...",20.8,2.4,5,0


In practice, we convert the numeric role into a name (start, hand, foot, or finish). This does not compromise model training because we will use a multi-hot encoding strategy.

In [3]:
ROLE_MAP = {12: "start", 13: "hand", 14: "finish", 15: "foot"}
ROLES = list(ROLE_MAP.values())  # ["start", "hand", "finish", "foot"]
def parse_holds(holds_str):
    """holds_str: stringified list of [hold_id, x, y, role_id]."""
    if isinstance(holds_str, str):
        holds_str = ast.literal_eval(holds_str)
    # Convert numeric role -> name once here, rest of pipeline unchanged
    return [[h[0], h[1], h[2], ROLE_MAP[h[3]]] for h in holds_str]
    
df_tmp = df.copy(deep=True)
df_tmp["holds"] = df_tmp["holds"].apply(parse_holds)
display(df_tmp.head(2))

,climb_uuid,name,angle,layout_id,n_holds,holds,difficulty_average,quality_average,ascensionist_count,is_nomatch
0,167e313e708e42e39796b521ca6eeec7,pogo a gogo,45,1,11,"[[1097, 64, 16, foot], [1137, 112, 32, foot], ...",21.0,2.6,5,0
1,F1DFFDBB7EB94D65AADAA43B7D817836,foot secret,0,1,14,"[[1076, 112, 8, foot], [1079, 88, 8, foot], [1...",20.8,2.4,5,0


## 2. Multi-hot encoding of holds and roles

A feature is created for each `(hold, role)` pair. To reduce the dimensionality, we require a minimum frequency (`min_freq`) for observing a `(hold, role)` pair in the dataset.

In [4]:
def build_multihot_features(df: pd.DataFrame, min_freq: int = 5) -> pd.DataFrame:
    """One column per (hold_id, role) combo used at least `min_freq` times."""

    # Count frequency of each (hold_id, role) across dataset
    counts = {}
    for holds in df["holds"]:
        for hold in holds:
            hold_id, x, y, role = hold
            key = (hold_id, role)
            counts[key] = counts.get(key, 0) + 1

    valid_keys = [k for k, v in counts.items() if v >= min_freq]
    key_to_col = {k: f"hold_{k[0]}_{k[1]}" for k in valid_keys}

    # Store each retained (hold, role) pair as a binary indicator column.
    multihot = pd.DataFrame(0, index=df.index, columns=list(key_to_col.values()))
    for idx, holds in zip(df.index, df["holds"]):
        for hold in holds:
            hold_id, x, y, role = hold
            key = (hold_id, role)
            if key in key_to_col:
                multihot.at[idx, key_to_col[key]] = 1

    return multihot
df_tmp = df.copy(deep=True)
df_tmp["holds"] = df_tmp["holds"].apply(parse_holds)
multihot_feats = build_multihot_features(df_tmp, min_freq=5)
df_tmp = pd.concat(
        [df[["difficulty_average", "ascensionist_count", "angle", "n_holds", "is_nomatch"]],
          multihot_feats],
        axis=1,
    )
display(df_tmp.head(2))

,difficulty_average,ascensionist_count,angle,n_holds,is_nomatch,hold_1097_foot,hold_1137_foot,hold_1148_foot,hold_1153_start,hold_1201_start,hold_1222_hand,hold_1238_hand,hold_1303_hand,hold_1321_hand,hold_1355_finish,hold_1532_foot,hold_1076_foot,hold_1079_foot,hold_1137_start,hold_1148_hand,hold_1186_hand,hold_1285_hand,hold_1392_finish,hold_1480_foot,hold_1488_foot,hold_1513_foot,hold_1514_foot,hold_1543_foot,hold_1553_foot,hold_1588_hand,hold_1117_foot,hold_1131_foot,hold_1154_start,hold_1157_foot,hold_1169_foot,hold_1199_foot,hold_1201_hand,hold_1202_start,hold_1266_hand,hold_1387_finish,...,hold_1270_foot,hold_1298_foot,hold_1274_start,hold_4773_hand,hold_1483_start,hold_4744_start,hold_1379_start,hold_4748_hand,hold_1280_start,hold_4811_foot,hold_4740_foot,hold_4709_hand,hold_1143_foot,hold_4738_foot,hold_1272_start,hold_1330_foot,hold_1337_finish,hold_4721_foot,hold_4817_foot,hold_4799_foot,hold_4776_foot,hold_1278_start,hold_1273_start,hold_4722_foot,hold_1276_start,hold_1317_foot,hold_1225_foot,hold_1119_foot,hold_1103_start,hold_1495_start,hold_1395_foot,hold_1347_foot,hold_4719_foot,hold_1530_start,hold_1094_start,hold_1381_hand,hold_1281_start,hold_1599_foot,hold_4816_foot,hold_4704_start
0,21.0,5,45,11,0,1,1,1,1,1,1,1,1,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,20.8,5,0,14,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,1,1,1,1,1,1,1,1,1,1,1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


## 3. Geometric features

Since we extracted the `(x, y)` position of every hold in the previous notebook, we can enrich the dataset with hand-crafted spatial descriptors of each climb. This provides direct information on how the holds are arranged in space. It is also an opportunity to point out to the models some important factors driving the difficulty of a climb: how far apart the holds are, how rare the foot holds are, whether there is a big dynamic move, etc.

These features are chosen from my personal climbing experience on the Kilter board, and the exploratory analysis already supports the choice: `avg_consecutive_move`, the average distance between consecutive hand holds, shows a high correlation factor with the grade — higher than `angle` itself.

Some comments are in order:
- Distances between consecutive holds are calculated using a proxy ordering obtained by sorting holds along the vertical axis. This is imperfect for climbs with horizontal traverses, but reasonable overall. In particular, `max_consecutive_move` acts as a **crux** proxy: the single biggest move of the climb.
-  We add **interaction features** that scale the key distances by the board `angle` (e.g., `move_dist_x_angle`, `avg_consecutive_move_x_angle`, and `max_foot_dist_x_angle`), because the difficulty of a given physical reach increases significantly as the wall gets steeper.

In [5]:
# ---------- Geometric features ----------

def euclidean(p1, p2):
    return np.sqrt((p1[0] - p2[0]) ** 2 + (p1[1] - p2[1]) ** 2)


def build_geometric_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Hand-crafted spatial/statistical descriptors of each climb.

    """
    records = []
    for holds, angle in zip(df["holds"], df["angle"]):
        by_role = {r: [(x, y) for _, x, y, role in holds if role == r] for r in ROLES}
        all_pts = [(x, y) for _, x, y, _ in holds]

        # Hand-grabbed holds: hand + start + finish (all touched by hand)
        hand_pts = by_role["hand"] + by_role["start"] + by_role["finish"]
        hand_sorted = sorted(hand_pts, key=lambda p: p[1])  # bottom to top (proxy order)

        xs, ys = zip(*all_pts) if all_pts else ([0], [0])
        board_width = max(xs) - min(xs) if all_pts else 0
        board_height = max(ys) - min(ys) if all_pts else 0

        # All pairwise distances between hand holds (global spread measure)
        dists = (
            [euclidean(a, b) for a, b in itertools.combinations(hand_pts, 2)]
            if len(hand_pts) > 1 else [0]
        )
        
        # All pairwise distances between foot holds
        foot_dists = [euclidean(a,b) for a,b in itertools.combinations(by_role["foot"], 2)] or [0]


        # Move distances in climbing order 
        # proxy ordering : only consecutive ones along the vertical ordering
        consecutive_dists = (
            [euclidean(a, b) for a, b in zip(hand_sorted, hand_sorted[1:])]
            or [0]
        )


        records.append({
            # --- counts ---
            "n_hand": len(by_role["hand"]),
            "n_foot": len(by_role["foot"]),
            "n_holds_total": len(all_pts),

            # --- board geometry ---
            "board_width": board_width,
            "board_height": board_height,
            "hold_density": len(all_pts) / max(board_width * board_height, 1),

            # --- hand holds distances (global spread) ---
            "avg_move_dist": np.mean(dists),
            "max_move_dist": np.max(dists),
            "std_move_dist": np.std(dists),
            
            # --- foot holds ---
            "avg_foot_dist": np.mean(foot_dists),
            "max_foot_dist": np.max(foot_dists),
            "std_foot_dist": np.std(foot_dists),

            # --- foot/hand relationship ---
            "foot_to_hand_ratio": len(by_role["foot"]) / max(len(hand_pts), 1),
            "hand_foot_dist": (
                euclidean(
                    (np.mean([p[0] for p in hand_pts]), np.mean([p[1] for p in hand_pts])),
                    (np.mean([p[0] for p in by_role["foot"]]), np.mean([p[1] for p in by_role["foot"]])),
                )
            ) if by_role["foot"] else 0,

            # --- consecutive (proxy-ordered) move distances ---
            "avg_consecutive_move": np.mean(consecutive_dists),
            "max_consecutive_move": np.max(consecutive_dists),  # crux proxy
            "std_consecutive_move": np.std(consecutive_dists),


            # --- overall climb span ---
            "start_finish_dist": (
                euclidean(
                    (np.mean([p[0] for p in by_role["start"]]), np.mean([p[0] for p in by_role["start"]])),
                    (np.mean([p[0] for p in by_role["finish"]]), np.mean([p[1] for p in by_role["finish"]])),
                )
            ) if by_role["finish"] and by_role["start"] else 0,
            
            
            # Interactions with angle: 
            "move_dist_x_angle": np.max(consecutive_dists) * angle, #crux move distance scaled by wall angle.
            "avg_consecutive_move_x_angle": np.mean(consecutive_dists) * angle,
            "max_foot_dist_x_angle": np.max(foot_dists) * angle,
        })

    return pd.DataFrame(records, index=df.index)
df["holds"] = df["holds"].apply(parse_holds)
geo_feats = build_geometric_features(df)
df = pd.concat(
    [df[["difficulty_average", "ascensionist_count", "angle", "n_holds", "is_nomatch"]],
    geo_feats, multihot_feats],
    axis=1,
    )
display(df.head(2))

,difficulty_average,ascensionist_count,angle,n_holds,is_nomatch,n_hand,n_foot,n_holds_total,board_width,board_height,hold_density,avg_move_dist,max_move_dist,std_move_dist,avg_foot_dist,max_foot_dist,std_foot_dist,foot_to_hand_ratio,hand_foot_dist,avg_consecutive_move,max_consecutive_move,std_consecutive_move,start_finish_dist,move_dist_x_angle,avg_consecutive_move_x_angle,max_foot_dist_x_angle,hold_1097_foot,hold_1137_foot,hold_1148_foot,hold_1153_start,hold_1201_start,hold_1222_hand,hold_1238_hand,hold_1303_hand,hold_1321_hand,hold_1355_finish,hold_1532_foot,hold_1076_foot,hold_1079_foot,hold_1137_start,...,hold_1270_foot,hold_1298_foot,hold_1274_start,hold_4773_hand,hold_1483_start,hold_4744_start,hold_1379_start,hold_4748_hand,hold_1280_start,hold_4811_foot,hold_4740_foot,hold_4709_hand,hold_1143_foot,hold_4738_foot,hold_1272_start,hold_1330_foot,hold_1337_finish,hold_4721_foot,hold_4817_foot,hold_4799_foot,hold_4776_foot,hold_1278_start,hold_1273_start,hold_4722_foot,hold_1276_start,hold_1317_foot,hold_1225_foot,hold_1119_foot,hold_1103_start,hold_1495_start,hold_1395_foot,hold_1347_foot,hold_4719_foot,hold_1530_start,hold_1094_start,hold_1381_hand,hold_1281_start,hold_1599_foot,hold_4816_foot,hold_4704_start
0,21.0,5,45,11,0,4,4,11,48,120,0.001910,46.621929,97.32420,22.814271,46.540798,61.188234,12.407823,0.571429,50.341284,24.258898,40.000000,11.697829,44.181444,1800.0,1091.650406,2753.470537,1,1,1,1,1,1,1,1,1,1,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,20.8,5,0,14,0,4,8,14,64,144,0.001519,69.393431,121.85237,29.924656,50.016530,91.389277,21.194431,1.333333,37.666667,41.514322,53.665631,9.020036,40.000000,0.0,0.000000,0.000000,0,0,0,0,0,0,0,0,0,0,0,1,1,1,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
